<a href="https://colab.research.google.com/github/keertiam8/text-classification/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers --quiet
!pip install opendatasets --quiet

import opendatasets as od
od.download("https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: keertiam
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection


100%|██████████| 3.30M/3.30M [00:00<00:00, 95.6MB/s]

In [ ]:
import torch
from torch import nn
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cuda


In [ ]:
data_df = pd.read_json('/content/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json', lines=True)
data_df.dropna(inplace = True)
data_df.drop(["article_link"], inplace = True, axis = 1)
print(data_df.shape)
data_df.head()


(26709, 2)


,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


In [ ]:
X_train,X_test, y_train, y_test = train_test_split((data_df["headline"]), np.array(data_df["is_sarcastic"]), test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5)

print(X_train.shape[0], "which is:", round(X_train.shape[0]/data_df.shape[0],4)*100,"%")
print(X_val.shape[0], "which is:", round(X_val.shape[0]/data_df.shape[0],4)*100,"%")
print(X_test.shape[0], "which is:", round(X_test.shape[0]/data_df.shape[0],4)*100,"%")


18696 which is: 70.0 %
4006 which is: 15.0 %
4007 which is: 15.0 %


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
class dataset(Dataset):
  def __init__(self,X,Y):
    self.X = [tokenizer(x,max_length=100,truncation=True,padding="max_length",return_tensors = 'pt').to(device) for x in X]
    self.Y = torch.tensor(Y,dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.X)
  def __getitem__(self,indx):
    return self.X[indx], self.Y[indx]

training_data = dataset(X_train, y_train)
validation_data = dataset(X_val,y_val)
testing_data = dataset(X_test,y_test)

In [ ]:
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4


In [ ]:
training_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=True)
testing_loader = DataLoader(testing_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class MyModel(nn.Module):
  def __init__(self,bert):
    super(MyModel,self).__init__()
    self.bert=bert
    self.dropout = nn.Dropout(0.25)
    self.linear1 = nn.Linear(768,384)
    self.linear2 = nn.Linear(384,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, input_ids, attention_mask):
    pooled_output = self.bert(input_ids, attention_mask,return_dict = False)[0][:,0]
    output = self.linear1(pooled_output)
    output = self.dropout(output)
    output = self.linear2(output)
    output = self.sigmoid(output)
    return output

In [ ]:
for param in bert_model.parameters():
  param.requires_grad = False
model = MyModel(bert_model).to(device)



In [ ]:
model

MyModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [ ]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=LR)

In [ ]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot =[]

for epoch in range(EPOCHS):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_val = 0
  total_loss_val = 0



  for indx,data in enumerate(training_loader):
    inputs, labels = data
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to device
    labels = labels.to(device)

    input_ids = inputs["input_ids"].squeeze(1)
    attention_mask = inputs["attention_mask"].squeeze(1)

    prediction = model(input_ids,attention_mask)
    prediction = prediction.squeeze(1)
    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()


    acc = (prediction.round() == labels).sum().item()

    total_acc_train += acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for indx,data in enumerate(validation_loader):
      inputs, labels = data
      inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to device
      labels = labels.to(device)

      input_ids = inputs["input_ids"].squeeze(1)
      attention_mask = inputs["attention_mask"].squeeze(1)

      prediction= model(input_ids,attention_mask)
      prediction = prediction.squeeze(1)
      batch_loss = criterion(prediction,labels)
      total_loss_val += batch_loss.item()


      acc = (prediction.round() == labels).sum().item()

      total_acc_val += acc

  total_loss_train_plot.append(total_loss_train/len(training_loader))
  total_loss_validation_plot.append(total_loss_val/len(validation_loader))
  total_acc_train_plot.append(total_acc_train/training_data.__len__()*100)
  total_acc_validation_plot.append(total_acc_val/validation_data.__len__()*100)

  print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/len(training_loader), 4)} Train Accuracy {round((total_acc_train)/training_data.__len__() * 100, 4)}%')
  print(f'Validation Loss: {round(total_loss_val/len(validation_loader), 4)} Validation Accuracy {round((total_acc_val)/validation_data.__len__() * 100, 4)}%')
  print()

Epoch 1/10, Train Loss: 2.5896 Train Accuracy 80.2739%

Epoch 2/10, Train Loss: 2.0982 Train Accuracy 84.7401%

Epoch 3/10, Train Loss: 1.9619 Train Accuracy 85.4621%

Epoch 4/10, Train Loss: 1.8969 Train Accuracy 86.0184%

Epoch 5/10, Train Loss: 1.8741 Train Accuracy 86.1361%



In [ ]:
with torch.no_grad():
  total_loss_test = 0
  total_acc_test = 0

  for indx,data in enumerate(testing_loader):
      inputs, labels = data
      inputs.to(device)
      labels.to(device)
      input_ids = inputs["input_ids"].squeeze(1)
      attention_mask = inputs["attention_mask"].squeeze(1)


      prediction = model(input_ids,attention_mask)
      prediction = prediction.squeeze(1)
      batch_loss = criterion(prediction,labels)
      total_loss_val += batch_loss.item()

      acc = (prediction.round() ==  labels).sum().item()
      total_acc_test += acc


print(f"accuracy score : {round(total_acc_test/testing_data.__len__()*100,4)}")
